# Lab 2 — Test-Driven Fix Loop

**Goal:** Wire an LLM into the sandbox to form the core write → test → fix loop.  
This is the engine behind tools like Devin and GitHub Copilot Autofix.

Pattern:
```
LLM writes solution
    ↓
pytest runs
    ↓ fail
LLM reads error + current code → generates fix
    ↓
pytest runs again ... until pass or budget exhausted
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from code_agent import run_code_agent
print('Code agent loaded ✓')

## 1. Simple example — FizzBuzz

In [ ]:
PROBLEM = """
Write a function fizzbuzz(n: int) -> list[str] that returns a list for numbers 1..n:
- 'FizzBuzz' if divisible by both 3 and 5
- 'Fizz' if divisible by 3 only  
- 'Buzz' if divisible by 5 only
- The number as string otherwise
"""

TESTS = """
def test_basic():
    r = fizzbuzz(15)
    assert r[0] == '1'
    assert r[2] == 'Fizz'
    assert r[4] == 'Buzz'
    assert r[14] == 'FizzBuzz'

def test_length():
    assert len(fizzbuzz(10)) == 10
"""

run = run_code_agent(PROBLEM, TESTS, verbose=True)
print('\nFinal solution:')
print(run.solution)

## 2. Harder example — Binary search
The agent will likely make a mistake first, then self-correct.

In [ ]:
PROBLEM2 = """
Write a function binary_search(arr: list[int], target: int) -> int
that returns the INDEX of target in the sorted list arr,
or -1 if not found. Must run in O(log n).
"""

TESTS2 = """
def test_found():
    assert binary_search([1, 3, 5, 7, 9], 5) == 2

def test_first():
    assert binary_search([1, 3, 5, 7, 9], 1) == 0

def test_last():
    assert binary_search([1, 3, 5, 7, 9], 9) == 4

def test_not_found():
    assert binary_search([1, 3, 5, 7, 9], 4) == -1

def test_empty():
    assert binary_search([], 1) == -1
"""

run2 = run_code_agent(PROBLEM2, TESTS2, verbose=True)
print('\nPassed:', run2.passed, '| Iterations:', run2.iterations, '| Cost: $', f'{run2.total_cost_usd:.4f}')

## 3. Inspect iteration history
See exactly what the agent generated at each step.

In [ ]:
for i, entry in enumerate(run2.history):
    print(f'--- Iteration {i+1} ({entry["role"]}) ---')
    print(entry['content'][:300])
    print()

## 4. Cost and token tracking

In [ ]:
print(f'Total tokens used: {run2.total_tokens:,}')
print(f'Estimated cost:    ${run2.total_cost_usd:.5f}')
print(f'Cost per iteration: ${run2.total_cost_usd / run2.iterations:.5f}')

## Exercise
Give the agent a deliberately broken problem description (e.g., 'sort in ascending order' but test expects descending).  
Observe: does it ever pass? What does the loop look like when it hits max_iterations?

**Hint:** Set `max_iterations=3` so it fails fast.

In [ ]:
# Your code here
